# Tutorial n°5

## Monte Carlo exploration

### A. Monte-Carlo exploration

In [ ]:
import deMonPy

deMonPy.configure_from_file("../global.json")

In [ ]:
from deMonPy.molden import read_XYZ

images, comm = read_XYZ("./data/MOLECULES")

WAT = images[0]
BZZ = images[1]

In [ ]:
import copy
import shutil

import numpy as np

from deMonPy.deMonNano import deMonNano

WORKDIR = ".run/tutorial-5/A/"

parameters_ptmc = {
    "DEMON_EXECUTABLE": deMonPy.DEMON_EXECUTABLE,
    "BASIS": {"PTYPE": "MAT", "SKFILE": deMonPy.DEMON_BASIS},
    "DEMON_PARAMETERS": {
        "ACTIVE": {
            "DFTB": {"SCC": True, "DISP": 2},
            "MOLECULES": {"NAMES": ["WAT", "BZZ", "WAT"]},
            "QUATERNION": {
                "RIGID": True,
                "COORDS": np.array(
                    [
                        [1.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0],
                        [0.0, -3.0, 0.0, 0.7, 0.7, 0.0, 0.0],
                        [1.0, -3.0, -4.0, 1.0, 0.0, 0.0, 0.0],
                    ]
                ),
            },
        },
    },
    "DEMON_MODULE": {
        "ACTIVE": {
            "PTMC": {
                "MC": {"MAX": 100, "WALL": 6.0},
                "MCTEMP": {
                    "TMC": 300,
                    "NTEMP": 1,
                },
            }
        }
    },
}

parameters = copy.deepcopy(parameters_ptmc)

dem = deMonNano(
    title="CALCULATION DEMONANO", workdir=WORKDIR, **(parameters_ptmc.copy())
)

# Move MOLECULES from data/ repository
shutil.copy2("./data/MOLECULES", f"{WORKDIR}/MOLECULES")

dem.calculate(
    symbols=WAT.symbols,  # IGNORE -> MOLECULES
    positions=WAT.positions,  # IGNORE -> MOLECULES
    clean_repository=False,
)

### B. Parallel tempering

In [ ]:
WORKDIR = ".run/tutorial-5/B/"

params = parameters.copy()
params["DEMON_MODULE"]["ACTIVE"].update(
    {
        "PTMC": {
            "MC": {"MAX": 500, "WALL": 6.0},
            "MCTEMP": {
                "GEOM": True,
                "NTEMP": 10,
                "TEMPMIN": 40,
                "TEMPMAX": 300,
                "SMOD": 20,
                "SDBG": True,
            },
        }
    }
)

dem = deMonNano(title="CALCULATION DEMONANO", workdir=WORKDIR, **params)

# Move MOLECULES from data/ repository
shutil.copy2("./data/MOLECULES", f"{WORKDIR}/MOLECULES")

dem.calculate(
    symbols=WAT.symbols,  # IGNORE -> MOLECULES
    positions=WAT.positions,  # IGNORE -> MOLECULES
    clean_repository=True,
)

In [ ]:
import os

import matplotlib.pyplot as plt

src = os.path.join(WORKDIR, "debug_swap.dat")

with open(src, "r") as fd:
    lines = fd.readlines()

table = []
for line in lines:
    table.append([float(elment) for elment in line.split()])

table = np.array(table)

for i in range(1, 11):
    plt.plot(table[:, i])
plt.show()

### C. MEMO-SCC parallel tempering

In [ ]:
WORKDIR = ".run/tutorial-5/C/"

params = parameters.copy()
params["DEMON_MODULE"]["ACTIVE"].update(
    {
        "PTMC": {
            "MC": {"MAX": 500, "WALL": 6.0},
            "MCTEMP": {
                "GEOM": True,
                "NTEMP": 10,
                "TEMPMIN": 40,
                "TEMPMAX": 300,
                "SMOD": 7,
                "SPERCENT": 100,
            },
        }
    }
)

dem = deMonNano(title="CALCULATION DEMONANO", workdir=WORKDIR, **params)

# Move MOLECULES from data/ repository
shutil.copy2("./data/MOLECULES", f"{WORKDIR}/MOLECULES")

dem.calculate(
    symbols=WAT.symbols,  # IGNORE -> MOLECULES
    positions=WAT.positions,  # IGNORE -> MOLECULES
    clean_repository=True,
)